In [7]:
# implement a fully connected network as a warm up
import torch
from torch import nn
import numpy as np

# Fully connected

In [19]:
class FullyConnected(nn.Module):
    def __init__(self, d_in, d_out, d_h=1024):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.d_h = d_h
        self.fcn_1 = nn.Linear(self.d_in, self.d_h)
        self.gelu = nn.GELU()
        self.fcn_2 = nn.Linear(self.d_h, self.d_out)

    def forward(self, x):
        batch_size, d_in =  x.shape
        x = self.fcn_1(x)
        x = self.gelu(x)
        x = self.fcn_2(x)
        return x

In [33]:
input_size = 10
n_sample = 500
x = torch.rand(n_sample, input_size)
y = torch.randint(0, 2, (n_sample,))

In [34]:
fully_connected = FullyConnected(input_size, 2) # if it is classification, output dimension should be 2

In [35]:
y_pred = fully_connected(x)

In [30]:
y.shape

torch.Size([500])

In [31]:
y_pred.shape

torch.Size([500, 2])

In [38]:
learning_rate = 1e-3
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(fully_connected.parameters(), lr = learning_rate)

In [39]:
dataset = torch.utils.data.TensorDataset(x, y)

In [40]:
batch_size = 32
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [42]:
# training loop
n_epoch = 100
for epoch in range(n_epoch):
    running_loss = 0
    for input, target in dataloader:
        optimizer.zero_grad()
        loss = loss_function(fully_connected(input), target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    if (epoch+1)%10 == 0:
        print(f'Epoch: {epoch+1}/{n_epoch}, running loss: {running_loss/len(dataloader):.4f}')

Epoch: 10/100, running loss: 0.6327
Epoch: 20/100, running loss: 0.6279
Epoch: 30/100, running loss: 0.6384
Epoch: 40/100, running loss: 0.6246
Epoch: 50/100, running loss: 0.6325
Epoch: 60/100, running loss: 0.6227
Epoch: 70/100, running loss: 0.6244
Epoch: 80/100, running loss: 0.6221
Epoch: 90/100, running loss: 0.6235
Epoch: 100/100, running loss: 0.6113


In [46]:
#inference
x_test = torch.rand(20, input_size)
y_test = torch.randint(0, 2, (20,))

In [50]:
with torch.no_grad():
    y_pred = fully_connected(x_test)

In [51]:
y_pred.shape

torch.Size([20, 2])

In [56]:
_, y_pred = torch.max(y_pred, 1)

In [57]:
y_pred

tensor([0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0])

# Transformer

In [69]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0, "d_model is not integer multiplier of n_head!"
        self.d_model = d_model
        self.d_head = d_model // n_head
        self.n_head = n_head
        self.scale = self.d_head ** -0.5

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, is_causal=False):
        # dim: (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape

        # q, k, v dim: (batch_size, n_head, seq_len, d_head)
        q = self.W_q(x).reshape(batch_size, seq_len, self.n_head, self.d_head).transpose(1,2).contiguous()
        k = self.W_k(x).reshape(batch_size, seq_len, self.n_head, self.d_head).transpose(1,2).contiguous()
        v = self.W_v(x).reshape(batch_size, seq_len, self.n_head, self.d_head).transpose(1,2).contiguous()

        # attn dim: (batch_size, n_head, seq_len, seq_len)
        attn = q @ k.transpose(-1, -2) * self.scale
        
        if is_causal:
            causal_mask = torch.triu(torch.ones_like(attn, device=x.device, dtype=torch.bool), diagonal=1)
            attn.masked_fill(causal_mask, -torch.inf)

        norm_attn = torch.softmax(attn, dim=-1)
        output = (norm_attn @ v).transpose(1,2).reshape(batch_size, seq_len, self.d_model).contiguous()
        output = self.out_proj(output)
        return output

In [70]:
x = torch.rand(2, 25, 48)

In [71]:
self_attn = SelfAttention(48, 8)

In [72]:
output = self_attn(x)

In [73]:
output.shape

torch.Size([2, 25, 48])